<a href="https://colab.research.google.com/github/ismoil27/deep_vision_mastery/blob/main/DocumentReader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image
from collections import defaultdict, Counter

import random
import os

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
print('device:', device)

device: cuda


In [4]:
CLASSES = ['Receipts', 'Others']
CLASS_TO_IDX = {'Receipts': 0, 'Others': 1}
NUM_CLASSES = 2

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.299, 0.224, 0.225]
    )
])


class DocumentDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.samples = []
        self.transform = transform

        for folder in os.listdir(root_dir):
            folder_path = os.path.join(root_dir, folder)

            if not os.path.isdir(folder_path):
                continue

            if folder == "Receipts":
                label = 0
            else:
                label = 1

            for img_name in os.listdir(folder_path):
                if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                    img_path = os.path.join(folder_path, img_name)
                    self.samples.append((img_path, label))


    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label


In [5]:
data_dir = '/content/drive/MyDrive/DCV'
dataset = DocumentDataset(data_dir, transform=transform)

print('Total samples BEFORE balance:', len(dataset))

Total samples BEFORE balance: 4372


In [6]:
# Class distribution
labels = [label for _, label in dataset.samples]
count = Counter(labels)

print('\n Before balance:')
print(f'Receipts: {count[0]}')
print(f'Others: {count[1]}')


class_samples = defaultdict(list)

for path, label in dataset.samples:
  class_samples[label].append((path, label))

min_count = min(len(class_samples[0]), len(class_samples[1]))
print('\n Using per class:', min_count)

balanced_samples = []

for label in class_samples:
  random.shuffle(class_samples[label])
  balanced_samples.extend(class_samples[label][:min_count])

dataset.samples = balanced_samples
dataset.targets = [label for _, label in balanced_samples]

print('Total samples AFTER balance:', len(dataset))


 Before balance:
Receipts: 438
Others: 3934

 Using per class: 438
Total samples AFTER balance: 876


In [7]:
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [8]:
model = resnet18(weights=ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


In [19]:
# freeze all layers
for param in model.parameters():
  param.requires_grad = False

# unfreeze last layer
for param in model.layer4.parameters():
  param.requires_grad = True


model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam([
    {"params": model.layer4.parameters(), "lr": 0.0001}, # lr => learning rate
    {"params": model.fc.parameters(), "lr": 0.001},
])

epochs = 10

for epoch in range(epochs):
  model.train()
  total_loss = 0

  for images, labels in train_loader:
    images = images.to(device)
    labels = labels.to(device)

    outputs = model(images)
    loss = criterion(outputs, labels)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

  # VALIDATION
  model.eval()
  correct = 0
  total = 0

  with torch.no_grad():
    for images, labels in val_loader:
      images = images.to(device)
      labels = labels.to(device)

      outputs = model(images)
      _, preds = torch.max(outputs, 1)

      correct += (preds == labels).sum().item()
      total += labels.size(0)

  acc = correct / total

  print(f'\n Epoch {epoch+1}')
  print(f' Loss: {total_loss:.4f}')
  print(f' Accuracy: {acc:.4f}')



# Save Model
torch.save(model.state_dict(), '/content/document_reader.pth')
print('\nModel saved!')


 Epoch 1
 Loss: 5.8632
 Accuracy: 0.9716

 Epoch 2
 Loss: 1.2549
 Accuracy: 0.9716

 Epoch 3
 Loss: 0.5589
 Accuracy: 0.9943

 Epoch 4
 Loss: 0.3732
 Accuracy: 0.9545

 Epoch 5
 Loss: 0.9488
 Accuracy: 0.9886

 Epoch 6
 Loss: 0.6164
 Accuracy: 0.9659

 Epoch 7
 Loss: 0.4171
 Accuracy: 1.0000

 Epoch 8
 Loss: 0.2997
 Accuracy: 0.9886

 Epoch 9
 Loss: 0.1447
 Accuracy: 0.9886

 Epoch 10
 Loss: 0.3747
 Accuracy: 0.9886

Model saved!
